# Pipeline Mini-4K → dataset de México → fine-tuning SAM-TP

Notebook orquestador. **No reimplementa nada**: llama a los scripts que ya están
en `ML_model/scripts/pipe_videos_online/`.

Guardalo en `ML_model/notebooks/` (o donde quieras: la celda 0 encuentra la raíz sola).

Orden: 0 setup → 1 metadata → 2 explorar → 3 select → 4 fetch → 5 clean → 6 curate → 7 chequeo.
Corré una celda a la vez y mirá la salida antes de seguir.


## 0 — Setup de rutas

El notebook vive en `ML_model/notebooks/`, asi que la raiz es `..`.
Todas las rutas salen de ahi. No cambia el directorio de trabajo.


In [ ]:
print("casa")

In [ ]:
import os, sys, json, subprocess
from pathlib import Path

# Este notebook vive en ML_model/notebooks/  ->  la raiz es la carpeta de arriba.
ML_ROOT   = Path.cwd().parent
PIPE_DIR  = ML_ROOT / "scripts" / "pipe_videos_online"
RIDES_DIR = ML_ROOT / "scripts" / "pipe_nuestras_rides"
DATA      = ML_ROOT / "data"
META_DIR  = ML_ROOT / "frodobots_metadata"      # gitignored
WORK      = DATA / "mexico"                     # todo lo de esta corrida vive aca
WORK.mkdir(parents=True, exist_ok=True)

# archivos que produce el pipeline, todos bajo WORK
SELECTED  = WORK / "selected_mexico.csv"
RAW_DIR   = WORK / "raw"
CLEAN_DIR = WORK / "clean"
INDEX     = WORK / "cleaned_index.csv"
CURATED   = WORK / "curated_mexico.csv"

sys.path.insert(0, str(PIPE_DIR))   # para importar Tool_2_Revisar_Parquet

assert PIPE_DIR.is_dir(), f"No existe {PIPE_DIR} -- corre el notebook desde ML_model/notebooks/"

for k, v in dict(ML_ROOT=ML_ROOT, PIPE_DIR=PIPE_DIR, META_DIR=META_DIR, WORK=WORK).items():
    print(f"{k:9s} = {v}   {'OK' if Path(v).exists() else '(no existe todavia)'}")


In [ ]:
def run(script: str, *args, cwd: Path = None) -> int:
    """Corre un script del repo mostrando la salida en vivo."""
    cwd = cwd or PIPE_DIR
    cmd = [sys.executable, script, *[str(a) for a in args]]
    print("$", " ".join(cmd), f"   (cwd={cwd})\n")
    p = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    p.wait()
    print(f"\n[exit {p.returncode}]")
    return p.returncode


## 1 — Metadata del Mini-4K

Baja solo `metadata.parquet` y archivos livianos (nada de video). Si ya lo tenés, saltea esta celda.

In [ ]:
from huggingface_hub import snapshot_download

if not any(META_DIR.rglob("*.parquet")):
    snapshot_download(
        repo_id="BitRobot/FrodoBots-Mini-4K",
        repo_type="dataset",
        allow_patterns=["meta/**", "**/*.json", "**/*.csv", "**/*.parquet"],
        ignore_patterns=["**/*.mp4", "**/*.tar", "**/*.avi", "**/*.mkv"],
        local_dir=str(META_DIR),
    )
else:
    print("Ya hay parquets en", META_DIR)

parquets = sorted(META_DIR.rglob("*.parquet"))
print(f"\n{len(parquets)} parquets:")
for p in parquets[:20]:
    print("  ", p.relative_to(META_DIR), f"{p.stat().st_size/1e6:.1f} MB")

META_PARQUET = next((p for p in parquets if p.name == "metadata.parquet"), parquets[0] if parquets else None)
print("\nMETA_PARQUET =", META_PARQUET)


## 2 — Explorar el metadata y encontrar el nombre exacto de México

Usa tu propio `Tool_2_Revisar_Parquet.explorar_parquet`. El filtro del `select`
es por coincidencia exacta, así que acá confirmás si dice `Mexico`, `México` o `MX`.

In [ ]:
from Tool_2_Revisar_Parquet import explorar_parquet

df_meta = explorar_parquet(META_PARQUET)

print("\n--- paises ---")
print(df_meta["country"].value_counts().to_string())


In [ ]:
# Elegí el valor EXACTO tal como aparece arriba
PAIS = "Mexico"     # <-- EDITAR si el value_counts dice otra cosa

mx = df_meta[df_meta["country"] == PAIS]
print(f"{len(mx)} rides en {PAIS}, {mx['db_dur_sec'].sum()/3600:.1f} h totales\n")

flags = [c for c in ["has_control","has_gps","has_imu","has_front_ts",
                     "has_rear_camera","has_video","complete"] if c in mx.columns]
print("Cuantos rides cumplen cada condicion:")
print(mx[flags].sum().to_string())

# cuantos pasan TODOS los filtros que exige el select
base = mx[flags[:4]].all(axis=1) if len(flags) >= 4 else None
if base is not None:
    print(f"\nPasan control+gps+imu+front_ts: {base.sum()}")
    if "has_rear_camera" in mx.columns:
        print(f"  ...y ademas tienen camara trasera: {(base & mx['has_rear_camera']).sum()}")


In [ ]:
# Inventario completo a Excel, por si lo querés mirar a mano
xlsx = WORK / "metadata_mexico.xlsx"
with __import__("pandas").ExcelWriter(xlsx, engine="openpyxl") as xl:
    mx.head(1_000_000).to_excel(xl, sheet_name="rides_mexico", index=False)
    df_meta["country"].value_counts().rename("n_rides").to_frame().to_excel(xl, sheet_name="por_pais")
print("->", xlsx)


## 3 — `select`: filtrar rides de México (no baja video)

Si el resultado te da pocos rides, sacá `--require-rear` y volvé a correr.

In [ ]:
REQUIRE_REAR = True    # <-- poné False si el paso 2 mostro pocos rides con camara trasera

args = ["--countries", PAIS, "--out", SELECTED]
if REQUIRE_REAR:
    args.append("--require-rear")

run("1_descarga_filtrado.py", "select", *args)


In [ ]:
import pandas as pd

sel = pd.read_csv(SELECTED)
print(f"{len(sel)} rides, {sel['db_dur_sec'].sum()/3600:.1f} h")
sel.to_excel(SELECTED.with_suffix(".xlsx"), index=False)
print("Excel ->", SELECTED.with_suffix(".xlsx"))
sel.head(20)


## 4 — `fetch`: bajar solo esos rides

**Este paso sí baja datos pesados.** Baja los shards que contienen los rides elegidos
y extrae solo esos. Empezá con pocos rides para medir cuánto tarda y cuánto ocupa.

In [ ]:
# Opcional: recortar a los N rides mas largos para una primera prueba
N_PRUEBA = 10          # <-- None para bajar todos los seleccionados

rides_file = SELECTED
if N_PRUEBA:
    rides_file = WORK / f"selected_mexico_top{N_PRUEBA}.csv"
    sel.head(N_PRUEBA).to_csv(rides_file, index=False)
    print(f"Usando los {N_PRUEBA} rides mas largos -> {rides_file}")

run("1_descarga_filtrado.py", "fetch", "--rides", rides_file, "--raw-dir", RAW_DIR)


In [ ]:
# Cuanto ocupo lo que bajaste
total = sum(p.stat().st_size for p in RAW_DIR.rglob("*") if p.is_file())
print(f"{RAW_DIR}: {total/1e9:.2f} GB en {len(list(RAW_DIR.iterdir()))} carpetas de ride")


## 5 — `clean`: sincronizar frame + GPS + control + IMU

Tolerancia de 500 ms, descarta GPS sin fix. Deja un `.synced.parquet` por ride
y un índice CSV.

In [ ]:
run("1_descarga_filtrado.py", "clean",
    "--raw-dir", RAW_DIR, "--clean-dir", CLEAN_DIR, "--index", INDEX)


In [ ]:
idx = pd.read_csv(INDEX)
print(idx.describe(include="all").T.to_string())
idx.head(20)


## 6 — `curate`: deduplicar por celda GPS

Evita quedarte con 30 rides de la misma cuadra. Para un primer fine-tune,
unas pocas horas bien diversas rinden más que muchas redundantes.

In [ ]:
HORAS = 20      # <-- objetivo de horas; bajalo para el primer fine-tune

run("1_descarga_filtrado.py", "curate",
    "--index", INDEX, "--hours", HORAS, "--out", CURATED)


In [ ]:
cur = pd.read_csv(CURATED)
cur.to_excel(CURATED.with_suffix(".xlsx"), index=False)
print(f"{len(cur)} rides curados -> {CURATED.with_suffix('.xlsx')}")
cur.head(20)


## 7 — Chequeo antes de etiquetar

Lo que sigue **no está en este notebook** porque necesita GPU y trabajo manual:

1. Extraer frames de los rides curados.
2. `pipe_nuestras_rides/2_evaluar_con_modelo.py` sobre esos frames
   (`--guardar-mascaras` para pre-anotar y solo corregir).
3. Corregir máscaras (CVAT / Label Studio / `2b_corregir_mascaras.py`).
4. `3_armar_dataset.py --labeled ... --out training/FOLD_custom --val-frac 0.15`.
5. Entrenar en el clone de `facebookresearch/sam2` con `sam2.1_custom2.yaml`,
   **partiendo de `checkpoint_finetuned_v2.pt`**, nunca de cero.

Antes de eso, dos cosas a resolver:

- **GPU.** El config de referencia es resolución 1024, batch 8. Una 2080 de 8 GB
  no lo aguanta: hay que bajar batch o buscar cluster.
- **IoU.** Hoy la validación es a ojo. Con datos de un país nuevo, el riesgo es
  olvido catastrófico, y sin métrica no lo ves. Está marcado como pendiente en
  `1_Doc tecnica y comandos.md`.


In [ ]:
print("Resumen de la corrida\n" + "="*40)
for nombre, ruta in [("metadata", META_DIR), ("selected", SELECTED), ("raw", RAW_DIR),
                     ("clean", CLEAN_DIR), ("index", INDEX), ("curated", CURATED)]:
    p = Path(ruta)
    estado = "OK" if p.exists() else "falta"
    print(f"  {nombre:9s} {estado:5s}  {p}")
